## Calculate Normalization Factor through csaw

In [ ]:
library(csaw)
library(edgeR)
library(BiocFileCache)
library(stringr)
library(Rsamtools)
library(edgeR)
options(repr.plot.width=15, repr.plot.height=8)

In [6]:
chip_meta <- read.csv('chip_meta.csv',sep = ',')
head(chip_meta)

,id,Pol,Ac,Het,celltype,sex,age_group,age,PMI.Hours,AgeGroup
,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>
1,390,,390-GABA-ac,,GABA,F,AGE_0,0.34000000,18.0,Infancy
2,390,,390-Glu-ac,,Glu,F,AGE_0,0.34000000,18.0,Infancy
3,4365,4365-GABA-pol,4365-GABA-ac,4365-GABA-het,GABA,M,AGE_0,0.03013699,8.7,Infancy
4,4365,4365-Glu-pol,4365-Glu-ac,4365-Glu-het,Glu,M,AGE_0,0.03013699,8.7,Infancy
5,4411,4411-GABA-pol,4411-GABA-ac,4411-GABA-het,GABA,M,AGE_0,0.15000000,18.0,Infancy
6,4411,4411-Glu-pol,4411-Glu-ac,4411-Glu-het,Glu,M,AGE_0,0.15000000,18.0,Infancy


In [7]:
blacklist <- rtracklayer::import.bed(
    "hg38-blacklist.v2.nochr.bed"
)

### Efficiency Bias

In [7]:
workers <- BiocParallel::MulticoreParam(workers=12)
filepath <- '/datasets/Public_Datasets/Dracheva_PsychEncode_development/processed/ChIP_GluGABA_filtered/paired'

# Initialize an empty data frame to store results
all_results_df <- data.frame()

for (celltype in c('Glu', 'GABA')) {
    for (assay in c('Ac', 'Pol', 'Het')) {

        chip_meta_sub <- chip_meta[chip_meta$celltype == celltype, ]


        chip_meta_sub <- chip_meta_sub[chip_meta_sub[[assay]] != '', ][[assay]]

    
        files <- paste0(chip_meta_sub,'.normchr.map60_as30_paired.bam')
        bam.files <- file.path(filepath, files)
        param <- readParam(minq=60, restrict= c(1:22,'X','Y'),discard = blacklist,pe="both")
        if (assay == 'Ac'){
            bin_size = 1000
        }
        else{
            bin_size = 10000
        }
        binned <- windowCounts(bam.files,  bin=TRUE, width=bin_size, param=param, BPPARAM=workers)
        me.bin <- windowCounts(bam.files, bin=TRUE, width=bin_size*10, param=param,BPPARAM=workers) 

        keep <- filterWindowsGlobal(binned, me.bin)$filter > log2(3)
        filtered.data <- binned[keep,]
        filtered.data<- normFactors(filtered.data)

        ids <- str_extract(files, "^\\d+")

        tmm_factor <- filtered.data$norm.factors
        
        lib_size <- asDGEList(binned)$samples$lib.size

        SizeFactors <- tmm_factor * lib_size / 1000000

        SizeFactors.Reciprocal <- 1/SizeFactors


        result_df <- data.frame(
            ID = ids,
            CellType = celltype,
            Assay = assay,
            TMM_normalization_factor = tmm_factor,
            lib_size = lib_size,
            SizeFactors = SizeFactors,
            SizeFactors.Reciprocal = SizeFactors.Reciprocal
        )

        # Append this iteration's results to the all_results_df
        all_results_df <- rbind(all_results_df, result_df)
    }
}

# Write the final aggregated DataFrame to a TSV file
file_path <- 'csaw_normalization/efficiency_biases_1_22XY_All_NormalizationFactors.tsv'
# write.table(all_results_df, file = file_path, sep = "\t", row.names = FALSE, quote = FALSE)